# Chapter 8: Reproducing a Published Result

<a href="https://colab.research.google.com/github/choilab-jefferson/medimage/blob/main/Chapter8_Reproducibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Every chapter so far ended by checking a number. This one checks a *paper*.

In 2014 Aerts and colleagues published one of the most cited results in radiomics: features extracted
from CT scans of lung tumours carry prognostic information about how long patients survive. The
analysis was done on a cohort known as **Lung1**, and both the images and the survival data are
public.

So the natural question is whether the result comes back out. Not "is the paper right" — that is a
different and much larger question — but the narrower and more useful one: **if I run a pipeline of
the same shape on the same public data, do I land somewhere near the published number?**

That question is worth more than it sounds. A pipeline that cannot reproduce a known result is not
ready to produce a new one.

By the end of this chapter you will have:

1. Run a complete radiomics study end to end, from download to survival statistics.
2. Compared your result against both a full-cohort reproduction and the published number.
3. Seen exactly how badly a small cohort behaves, and learned to recognise the symptoms.

## Setup

This chapter is driven by the `qr` command-line tool from
[qradiomics](https://github.com/choilab-jefferson/qradiomics), so most cells are shell commands
rather than Python.

Note the `opencv-python-headless` install. Converting radiotherapy contours needs `rt-utils`, which
pulls in OpenCV, and the ordinary OpenCV build requires a graphics library that servers and
containers do not have. Without the headless build every contour conversion fails with
`libGL.so.1: cannot open shared object file` — an error that looks nothing like its cause.

In [1]:
# --- Setup: Google Colab (primary) and local checkout (testing) ---------------
REPO_URL = "https://github.com/choilab-jefferson/medimage.git"
REPO_DIR = "medimage"

import os
import pathlib
import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    # On Colab the repository is not present yet, so clone it and install the
    # few packages that are not part of the default runtime.
    if not pathlib.Path(REPO_DIR).is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pydicom", "remotezip"],
        check=True,
    )
else:
    # Locally the notebook already lives inside the repository; walk up until
    # we find the module that the notebooks import.
    root = pathlib.Path.cwd().resolve()
    while not (root / "medimage_data.py").is_file() and root != root.parent:
        root = root.parent
    os.chdir(root)

print("Running on Colab:", IN_COLAB)
print("Working directory:", os.getcwd())

Running on Colab: False
Working directory: /home/wxc151/projects/Biomedical-Image-Analysis-in-Python


In [2]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "qradiomics", "rt-utils", "opencv-python-headless"], check=True)

print(subprocess.run(["qr", "info"], capture_output=True, text=True).stdout.strip())

qradiomics v0.9.0
Aliases: qradiomics, qr, qrdx
Verbose: False


In [3]:
import json
import os
import pathlib

import pandas as pd

WORK = pathlib.Path("work/lung1")
WORK.mkdir(parents=True, exist_ok=True)

# How many patients to use. The full cohort is 422. Survival signal is weak at
# small n, so raise this if you have the time and disk space; the download is
# roughly 25 MB per patient.
N_PATIENTS = 60

print(f"running on {N_PATIENTS} patients")

running on 60 patients


## 1. What is in the collection

`qr tcia series` lists every series in a public TCIA collection. We need patients who have **both**
a CT scan and a radiotherapy structure set, because the structure set is what tells us where the
tumour is.

In [4]:
series_csv = WORK / "series.csv"
if not series_csv.exists():
    subprocess.run(["qr", "tcia", "series", "--collection", "NSCLC-Radiomics",
                    "-o", str(series_csv)], check=True)

series = pd.read_csv(series_csv)
print(f"{len(series)} series")
print(series["Modality"].value_counts().to_string())

with_ct = set(series.loc[series.Modality == "CT", "PatientID"])
with_rt = set(series.loc[series.Modality == "RTSTRUCT", "PatientID"])
patients = sorted(with_ct & with_rt)[:N_PATIENTS]

print(f"\n{len(with_ct & with_rt)} patients have both; using {len(patients)}")

1265 series
Modality
CT          422
RTSTRUCT    422
SEG         421

422 patients have both; using 60


In [5]:
targets = series[series.PatientID.isin(patients)
                 & series.Modality.isin(["CT", "RTSTRUCT"])]
target_csv = WORK / "targets.csv"
targets.to_csv(target_csv, index=False)

subprocess.run(["qr", "tcia", "download", "--manifest", str(target_csv),
                "-o", str(WORK / "dicom"), "-j", "8"], check=True)

Will download 120 series → work/lung1/dicom  (workers=8, skip_existing=True)
  [    1/120]   0.8%  ok=0 cached=1 fail=0  (0 MB downloaded)  last=LUNG1-048 (CT)
  [    2/120]   1.7%  ok=0 cached=2 fail=0  (0 MB downloaded)  last=LUNG1-036 (CT)
  [    3/120]   2.5%  ok=0 cached=3 fail=0  (0 MB downloaded)  last=LUNG1-029 (CT)
  [    4/120]   3.3%  ok=0 cached=4 fail=0  (0 MB downloaded)  last=LUNG1-007 (CT)
  [    5/120]   4.2%  ok=0 cached=5 fail=0  (0 MB downloaded)  last=LUNG1-056 (CT)
  [    6/120]   5.0%  ok=0 cached=6 fail=0  (0 MB downloaded)  last=LUNG1-013 (CT)
  [    7/120]   5.8%  ok=0 cached=7 fail=0  (0 MB downloaded)  last=LUNG1-032 (CT)
  [    8/120]   6.7%  ok=0 cached=8 fail=0  (0 MB downloaded)  last=LUNG1-001 (CT)
  [    9/120]   7.5%  ok=0 cached=9 fail=0  (0 MB downloaded)  last=LUNG1-004 (CT)
  [   10/120]   8.3%  ok=0 cached=10 fail=0  (0 MB downloaded)  last=LUNG1-006 (CT)
  [   11/120]   9.2%  ok=0 cached=11 fail=0  (0 MB downloaded)  last=LUNG1-002 (CT)
  [   12

CompletedProcess(args=['qr', 'tcia', 'download', '--manifest', 'work/lung1/targets.csv', '-o', 'work/lung1/dicom', '-j', '8'], returncode=0)

## 2. DICOM to a form the feature extractor can read

Two conversions. The CT series becomes one volume file, and the structure set becomes a binary mask
marking the tumour.

**`--roi GTV-1` is not optional.** A structure set holds many contours — lungs, spinal cord,
oesophagus, treatment volumes — and without being told which one you want, the converter takes the
first. In Lung1 that is frequently a lung or the cord rather than the tumour. The pipeline runs
happily, produces features, and measures the wrong organ.

In [6]:
out_dir = WORK / "nrrd"
out_dir.mkdir(exist_ok=True)

ct_rows = targets[targets.Modality == "CT"]
rt_rows = targets[targets.Modality == "RTSTRUCT"]


def series_path(row):
    return WORK / "dicom" / row.PatientID / row.StudyInstanceUID / row.SeriesInstanceUID


converted = failed = 0
for _, row in ct_rows.iterrows():
    target = out_dir / f"{row.PatientID}_image.nrrd"
    if target.exists():
        converted += 1
        continue
    result = subprocess.run(["qr", "convert", "dicom-series",
                             "-i", str(series_path(row)), "-o", str(target)],
                            capture_output=True, text=True)
    converted += result.returncode == 0
    failed += result.returncode != 0

print(f"CT volumes: {converted} converted, {failed} failed")

CT volumes: 60 converted, 0 failed


In [7]:
converted = failed = 0
for _, row in rt_rows.iterrows():
    target = out_dir / f"{row.PatientID}_mask.nrrd"
    if target.exists():
        converted += 1
        continue
    ct_match = ct_rows[ct_rows.PatientID == row.PatientID]
    if ct_match.empty:
        continue
    result = subprocess.run(["qr", "convert", "rtstruct",
                             "-d", str(series_path(ct_match.iloc[0])),
                             "-r", str(series_path(row)),
                             "--roi", "GTV-1",
                             "-o", str(target)],
                            capture_output=True, text=True)
    converted += result.returncode == 0
    failed += result.returncode != 0

print(f"tumour masks: {converted} converted, {failed} failed")

tumour masks: 59 converted, 1 failed


## 3. Manifest, crop, extract

A **manifest** is the table that pairs each patient's image with their mask. Everything downstream
takes one.

`qr preprocess` then crops each volume to a margin around the tumour and resamples to 1 mm isotropic
voxels. The crop is for speed; the resampling is Chapter 4's lesson applied at scale — texture
features are computed over voxel neighbourhoods, so unless every patient is on the same voxel grid,
the same texture measured on two scanners produces two different numbers.

In [8]:
manifest = out_dir / "manifest.csv"
subprocess.run(["qr", "convert", "manifest-from-dir", "-d", str(out_dir),
                "--image-glob", "*_image.nrrd", "--mask-glob", "*_mask.nrrd",
                "-o", str(manifest)], check=True)

crop_dir = WORK / "cropped"
subprocess.run(["qr", "preprocess", "-m", str(manifest), "-o", str(crop_dir),
                "--pad-mm", "5", "--resample", "1.0", "--jobs", "4",
                "--out-manifest", str(crop_dir / "manifest.csv")], check=True)

Wrote manifest with 59 patient(s) -> work/lung1/nrrd/manifest.csv
  skipped 1 dir(s) without matching files (first 5): ['LUNG1-035']


Reading manifest: work/lung1/nrrd/manifest.csv  (59 patients, jobs=4)


  preprocess [1/59] LUNG1-003 ok
  preprocess [2/59] LUNG1-002 ok
  preprocess [3/59] LUNG1-004 ok
  preprocess [4/59] LUNG1-001 ok
  preprocess [5/59] LUNG1-005 ok
  preprocess [6/59] LUNG1-006 ok
  preprocess [7/59] LUNG1-007 ok
  preprocess [8/59] LUNG1-008 ok


  preprocess [9/59] LUNG1-009 ok
  preprocess [10/59] LUNG1-010 ok
  preprocess [11/59] LUNG1-011 ok
  preprocess [12/59] LUNG1-012 ok
  preprocess [13/59] LUNG1-014 ok
  preprocess [14/59] LUNG1-016 ok
  preprocess [15/59] LUNG1-013 ok
  preprocess [16/59] LUNG1-015 ok


  preprocess [17/59] LUNG1-020 ok
  preprocess [18/59] LUNG1-019 ok
  preprocess [19/59] LUNG1-018 ok
  preprocess [20/59] LUNG1-017 ok
  preprocess [21/59] LUNG1-024 ok
  preprocess [22/59] LUNG1-022 ok
  preprocess [23/59] LUNG1-023 ok
  preprocess [24/59] LUNG1-021 ok


...

  preprocess [40/59] LUNG1-041 ok
  preprocess [41/59] LUNG1-042 ok
  preprocess [42/59] LUNG1-044 ok
  preprocess [43/59] LUNG1-043 ok
  preprocess [44/59] LUNG1-045 ok
  preprocess [45/59] LUNG1-046 ok


  preprocess [46/59] LUNG1-047 ok
  preprocess [47/59] LUNG1-050 ok
  preprocess [48/59] LUNG1-049 ok
  preprocess [49/59] LUNG1-048 ok
  preprocess [50/59] LUNG1-053 ok
  preprocess [51/59] LUNG1-052 ok
  preprocess [52/59] LUNG1-051 ok


  preprocess [53/59] LUNG1-054 ok
  preprocess [54/59] LUNG1-057 ok
  preprocess [55/59] LUNG1-055 ok
  preprocess [56/59] LUNG1-056 ok


  preprocess [57/59] LUNG1-059 ok
  preprocess [58/59] LUNG1-060 ok
  preprocess [59/59] LUNG1-058 ok
Wrote cropped manifest → work/lung1/cropped/manifest.csv

Preprocess: 59 ok, 0 failed → work/lung1/cropped


CompletedProcess(args=['qr', 'preprocess', '-m', 'work/lung1/nrrd/manifest.csv', '-o', 'work/lung1/cropped', '--pad-mm', '5', '--resample', '1.0', '--jobs', '4', '--out-manifest', 'work/lung1/cropped/manifest.csv'], returncode=0)

`qr extract` computes the features. The `nsclc-survival` pattern reproduces the feature space the
paper used: the original image plus wavelet, Laplacian-of-Gaussian, square, square-root and
logarithm transforms, across seven feature classes — about 1130 numbers per patient.

Bear that number in mind. We are about to fit a survival model with **more features than patients**,
by a factor of nearly twenty.

In [9]:
features_csv = WORK / "features.csv"
if not features_csv.exists():
    subprocess.run(["qr", "extract", "-m", str(crop_dir / "manifest.csv"),
                    "-p", "nsclc-survival", "-o", str(features_csv), "-j", "4"], check=True)

features = pd.read_csv(features_csv)
print(f"{features.shape[0]} patients x {features.shape[1] - 1} features")

59 patients x 1130 features


## 4. Joining the survival data

The images tell you about the tumour. The clinical table tells you what happened to the patient.
`qr results merge` joins them and standardises the outcome into two columns: `OS_months` and
`OS_event`, where the event is 1 if the patient died during follow-up and 0 if they were still alive
when the study stopped recording.

That 0 does not mean "survived". It means "we stopped looking", and survival analysis exists
precisely to handle that distinction.

In [10]:
clinical_csv = WORK / "clinical.csv"
if not clinical_csv.exists():
    subprocess.run(["curl", "-sSL", "-o", str(clinical_csv),
                    "https://www.cancerimagingarchive.net/wp-content/uploads/"
                    "NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv"], check=True)

ready_csv = WORK / "analysis_ready.csv"
subprocess.run(["qr", "results", "merge", "-f", str(features_csv), "-c", str(clinical_csv),
                "--clinical-id-col", "PatientID",
                "--time-col", "Survival.time", "--event-col", "deadstatus.event",
                "-o", str(ready_csv)], check=True)

ready = pd.read_csv(ready_csv)
print(f"{len(ready)} patients, {ready.OS_event.sum()} deaths observed")
print(f"median follow-up: {ready.OS_months.median():.1f} months")

Merged 59 patients, 1133 columns -> work/lung1/analysis_ready.csv


59 patients, 54 deaths observed
median follow-up: 11.2 months


## 5. Which features look prognostic?

`qr analyze survival` fits a separate Cox proportional-hazards model for each feature and reports
its hazard ratio and p-value. A hazard ratio above 1 means higher values go with shorter survival.

In [11]:
cox_csv = WORK / "cox.csv"
subprocess.run(["qr", "analyze", "survival", "-i", str(ready_csv),
                "--outcome", "OS_months", "--event", "OS_event",
                "-o", str(cox_csv), "--top-n", "10"], check=True)

cox = pd.read_csv(cox_csv)
print(f"\n{len(cox)} features fitted, {(cox.p < 0.05).sum()} significant at p < 0.05")
print(f"expected by chance alone at p < 0.05: about {0.05 * len(cox):.0f}")

Running univariate Cox PH: 1130 features, 59 patients...



──────────────────────────────────────────────────────────────
Cox PH Univariate  |  1129 features  |  160 significant (p<0.05)
──────────────────────────────────────────────────────────────
  Feature                                          HR         p
──────────────────────────────────────────────────────────────
* wavelet-LLL_glszm_GrayLevelNonUniformity      1.002    0.0008
* wavelet-HHH_glszm_GrayLevelNonUniformity      1.028    0.0013
* wavelet-HHL_glszm_SizeZoneNonUniformity       1.001    0.0015
* wavelet-HHH_glszm_SizeZoneNonUniformity       1.017    0.0020
* wavelet-HLH_glszm_SmallAreaEmphasis          104386.712    0.0020
* wavelet-HHL_firstorder_TotalEnergy            1.000    0.0022
* wavelet-HHL_firstorder_Energy                 1.000    0.0022
* wavelet-HHH_firstorder_TotalEnergy            1.000    0.0024
* wavelet-HHH_firstorder_Energy                 1.000    0.0024
* square_glrlm_GrayLevelNonUniformity           1.000    0.0032
─────────────────────────────────────


1129 features fitted, 160 significant at p < 0.05
expected by chance alone at p < 0.05: about 56


Compare those two numbers before reading anything into the ranking.

Testing 1130 features at p < 0.05 produces about 56 "significant" results even if no feature carries
any signal at all. This is the multiple-comparisons problem, and in radiomics it is not a
technicality — it is the central difficulty. A single p-value from a list this long is close to
meaningless on its own.

In [12]:
top = cox.nsmallest(8, "p")[["feature", "hr", "p"]]
pd.set_option("display.width", 140)
print(top.to_string(index=False))

extreme = (cox.hr > 1e6) | (cox.hr < 1e-6)
print(f"\nfeatures with hazard ratios beyond 1e6 or below 1e-6: {extreme.sum()}")

                                 feature            hr        p
wavelet-LLL_glszm_GrayLevelNonUniformity      1.001886 0.000802
wavelet-HHH_glszm_GrayLevelNonUniformity      1.027920 0.001276
 wavelet-HHL_glszm_SizeZoneNonUniformity      1.000505 0.001511
 wavelet-HHH_glszm_SizeZoneNonUniformity      1.016869 0.002007
     wavelet-HLH_glszm_SmallAreaEmphasis 104386.712410 0.002026
      wavelet-HHL_firstorder_TotalEnergy      1.000000 0.002223
           wavelet-HHL_firstorder_Energy      1.000000 0.002223
      wavelet-HHH_firstorder_TotalEnergy      1.000000 0.002392

features with hazard ratios beyond 1e6 or below 1e-6: 132


Look at the size of those hazard ratios. A plausible clinical hazard ratio is something like 1.5 or
2. Values of $10^{20}$ or $10^{-100}$ are not findings — they are the arithmetic telling you that
the model separated the data perfectly because there were too few patients to constrain it.

**Absurd numbers are a symptom, and a useful one.** They are easier to notice than a quietly
overfitted model that produces believable-looking values.

## 6. The honest measure: cross-validated prediction

Univariate p-values say which features *look* associated with survival in the data you have. They do
not say whether a model built from them predicts anything about patients it has never seen.

For that you need cross-validation: fit on part of the cohort, predict the rest, repeat. The score
is the **concordance index (c-index)** — the probability that, given two patients, the model
correctly identifies which one died first.

- **0.5** — no better than a coin toss.
- **0.65** — the value Aerts reported on the full Lung1 cohort.
- **1.0** — perfect.

In [13]:
metrics_json = WORK / "metrics.json"
subprocess.run(["qr", "ml", "train", "-i", str(ready_csv),
                "--task", "survival", "--outcome", "OS_event", "--time-col", "OS_months",
                "--folds", "5", "--top-features", "30",
                "--model", str(WORK / "model.pkl"), "--metrics", str(metrics_json)], check=True)

metrics = json.loads(metrics_json.read_text())
c_index = metrics["cv_c_index_mean"]

print()
print(f"patients            : {metrics['n']}")
print(f"cross-validated c-index: {c_index:.3f} +/- {metrics['cv_c_index_std']:.3f}")
print(f"95% confidence interval: [{metrics['cv_c_index_ci_lo']:.3f}, "
      f"{metrics['cv_c_index_ci_hi']:.3f}]")

qr ml train: task=survival outcome=OS_event input=work/lung1/analysis_ready.csv
  feature reduction: corr<0.95, top-30
  loaded 59 rows × 1133 cols
  [survival] 59 rows, 1130 candidate features
  [survival] events=54/59


  [survival] cross-validating (5-fold, feature selection per fold)


    fold 1/5: c-index=0.415 (train=47, test=12, k=30)


    fold 2/5: c-index=0.561 (train=47, test=12, k=30)


    fold 3/5: c-index=0.439 (train=47, test=12, k=30)


    fold 4/5: c-index=0.484 (train=47, test=12, k=30)


    fold 5/5: c-index=0.509 (train=48, test=11, k=30)


  [survival] pooled c-index 95% CI = [0.377, 0.558]  IBS=0.246
  [survival] fitting final model on all 59 samples


  [survival] final feature set: 30 features
Model -> work/lung1/model.pkl
Metrics -> work/lung1/metrics.json
  CV c-index: 0.482 ± 0.051



patients            : 59
cross-validated c-index: 0.482 +/- 0.051
95% confidence interval: [0.377, 0.558]


## 7. The comparison

Three numbers for the same analysis at three different scales.

In [14]:
comparison = pd.DataFrame([
    {"run": "this notebook", "patients": metrics["n"],
     "c_index": round(c_index, 3), "source": "computed just now"},
    {"run": "full-cohort reproduction", "patients": 420,
     "c_index": 0.580, "source": "qradiomics reproducibility report"},
    {"run": "Aerts et al. 2014", "patients": 422,
     "c_index": 0.650, "source": "published"},
])

print(comparison.to_string(index=False))

                     run  patients  c_index                            source
           this notebook        59    0.482                 computed just now
full-cohort reproduction       420    0.580 qradiomics reproducibility report
       Aerts et al. 2014       422    0.650                         published


### Reading the table

**Our run finds nothing.** At sixty patients the cross-validated c-index comes out at roughly 0.48 —
marginally *below* chance — with a confidence interval that comfortably contains 0.5. Individual
folds scatter from about 0.42 to 0.56, which is the scatter you would get from shuffling the
outcomes.

It is worth being precise about what that means, because the temptation is to read it as a weak
positive result. It is not. A model that scores below 0.5 has not found a faint signal; it has found
none, and the small deviation is noise. With over a thousand features and fewer than sixty patients,
the model has enough freedom to fit the training folds perfectly and no reason for those fits to
carry over to the held-out patients.

**Do not tune your way out of this.** Faced with 0.48 it is easy to try a different feature count, a
different penalty, a different fold count, and stop when a run happens to produce 0.6. That number
would be a description of the search, not of the data. The honest conclusion from this run is that
sixty patients cannot answer the question.

**The full-cohort reproduction reaches 0.580 against a published 0.650.** That gap is a real and
common finding. Reproductions of radiomics results usually land below the original, and the reasons
are mundane rather than sinister: image preprocessing choices, which contour was used, feature
software versions, and how the model was selected. The published figure also came from a pipeline
tuned on that data, while a reproduction applies a fixed recipe.

**What survives is the direction, not the decimal.** Both reproductions land above 0.5, so the claim
that CT texture carries prognostic information holds. The claim that it carries *exactly* 0.65 worth
does not travel as well.

That distinction is the most useful thing in this chapter. "Does the effect reproduce?" and "does
the number reproduce?" are different questions, and confusing them is how radiomics acquired its
reputation for irreproducibility.

## 8. Raising the patient count

Change `N_PATIENTS` at the top and re-run. The cells are cached, so only the new patients are
downloaded and extracted. Watch what happens as it climbs:

| Patients | Measured here |
|---|---|
| 12 | c-index exactly 0.500, hazard ratios up to $10^{96}$ |
| 60 | c-index 0.48, CI [0.38, 0.56] — indistinguishable from chance |
| 150+ | not run here; expect the interval to start narrowing |
| 422 | 0.580 in the reproducibility report |

The first two rows were measured while writing this chapter. Nothing about the method changes
between them and the last row — only the amount of data. That alone is the difference between a
result and a coin toss.

## Limitations

- **Not the published pipeline.** Aerts et al. used a specific four-feature signature fitted their
  way. This is a pipeline of the same shape, not a line-by-line reimplementation, so exact agreement
  was never the goal.
- **No external validation.** The published claim was tested by transferring the signature to
  independent cohorts. Doing that here would need a second dataset.
- **The first N patients, not a random sample.** Convenient and reproducible, but a genuine study
  would sample properly.
- **One extraction setting.** Bin width, resampling and interpolation all move radiomics features,
  and this notebook fixes them without exploring the alternatives.

## Exercises

1. Set `N_PATIENTS` to 20 and then to 100 and record the c-index each time. Plot it against patient
   count. Where does it start to stabilise?
2. Re-run the extraction with `--bin-width 10` instead of the pattern default. How much do the
   feature values move? How much does the c-index move?
3. Convert a structure set *without* `--roi GTV-1` and compare the mask with the correct one using
   Chapter 3's Dice function. What organ did you actually get?
4. Of the features ranked most significant, how many would you still expect to be significant after
   correcting for 1130 tests (try Bonferroni: divide 0.05 by the number of tests)?

## References

- Aerts HJWL, Velazquez ER, Leijenaar RTH, et al. *Decoding tumour phenotype by noninvasive imaging
  using a quantitative radiomics approach.* Nature Communications. 2014;5:4006.
  [doi:10.1038/ncomms5006](https://doi.org/10.1038/ncomms5006)
- Aerts HJWL, Wee L, Rios Velazquez E, et al. (2019). *Data From NSCLC-Radiomics.*
  The Cancer Imaging Archive.
  [doi:10.7937/K9/TCIA.2015.PF0M9REI](https://doi.org/10.7937/K9/TCIA.2015.PF0M9REI)
- van Griethuysen JJM, Fedorov A, Parmar C, et al. *Computational radiomics system to decode the
  radiographic phenotype.* Cancer Research. 2017;77(21):e104–e107.
- Harrell FE, Califf RM, Pryor DB, et al. *Evaluating the yield of medical tests.* JAMA.
  1982;247(18):2543–2546. — the concordance index.